In [ ]:
import numpy as np
from scipy.optimize import differential_evolution, least_squares
U3_pmns = np.array([
    [ 0.821427,  0.550313,  0.149708],
    [-0.481513,  0.528538,  0.699138],
    [ 0.305618, -0.646377,  0.699138]
], dtype=float)

def embed_pmns_4x4(U3, phi4=0.0):
    U4 = np.eye(4, dtype=complex)
    U4[:3, :3] = U3
    U4[3, 3] = np.exp(1j * phi4)
    return U4

U4_target = embed_pmns_4x4(U3_pmns, phi4=0.0)

CNOT_AB = np.array([
    [1, 0, 0, 0],
    [0, 0, 0, 1],
    [0, 0, 1, 0],
    [0, 1, 0, 0]
], dtype=complex)

def u3_real(theta):
    return np.array([
        [np.cos(theta/2), -np.sin(theta/2)],
        [np.sin(theta/2),  np.cos(theta/2)]
    ], dtype=complex)

def layer(theta_a, theta_b):
    return np.kron(u3_real(theta_a), u3_real(theta_b))

def pmns_ansatz_4x4(params):
    alpha, beta, gamma, delta, eps, zeta = params
    return (
        layer(eps, zeta)
        @ CNOT_AB
        @ layer(gamma, delta)
        @ CNOT_AB
        @ layer(alpha, beta)
    )

def align_global_phase(U_fit, U_target):
    phase = np.angle(np.vdot(U_fit.flatten(), U_target.flatten()))
    return U_fit * np.exp(-1j * phase)

def fit_pmns_4x4(U_target, seed=1234):
    bounds = [(-np.pi, np.pi)] * 6

    def objective(x):
        U = align_global_phase(pmns_ansatz_4x4(x), U_target)
        return np.linalg.norm(U - U_target, ord='fro')**2

    de_result = differential_evolution(
        objective,
        bounds=bounds,
        seed=seed,
        polish=False,
        tol=1e-10
    )

    def residuals(x):
        U = align_global_phase(pmns_ansatz_4x4(x), U_target) - U_target
        return np.concatenate([U.real.ravel(), U.imag.ravel()])

    ls_result = least_squares(
        residuals,
        de_result.x,
        ftol=1e-14,
        xtol=1e-14,
        gtol=1e-14,
        max_nfev=50000
    )

    best_params = ls_result.x
    U_best = align_global_phase(pmns_ansatz_4x4(best_params), U_target)
    max_abs_err = np.max(np.abs(U_best - U_target))
    fro_err = np.linalg.norm(U_best - U_target, ord='fro')

    return best_params, U_best, max_abs_err, fro_err

params, U_fit, max_err, fro_err = fit_pmns_4x4(U4_target)

alpha, beta, gamma, delta, eps, zeta = params

print("Best-fit parameters:")
print(f"alpha   = {alpha:.8f}")
print(f"beta    = {beta:.8f}")
print(f"gamma   = {gamma:.8f}")
print(f"delta   = {delta:.8f}")
print(f"epsilon = {eps:.8f}")
print(f"zeta    = {zeta:.8f}")

print("\nTarget 4x4 PMNS embedding:")
print(np.round(U4_target, 8))

print("\nFitted 4x4 matrix:")
print(np.round(U_fit.real, 8))

print(f"\nMax |U_fit - U_target| = {max_err:.3e}")
print(f"Frobenius norm error   = {fro_err:.3e}")

Best-fit parameters:
alpha   = -0.60311799
beta    = -2.01246216
gamma   = -0.79660495
delta   = 1.01391924
epsilon = 0.70529519
zeta    = 1.35984944

Target 4x4 PMNS embedding:
[[ 0.821427+0.j  0.550313+0.j  0.149708+0.j  0.      +0.j]
 [-0.481513+0.j  0.528538+0.j  0.699138+0.j  0.      +0.j]
 [ 0.305618+0.j -0.646377+0.j  0.699138+0.j  0.      +0.j]
 [ 0.      +0.j  0.      +0.j  0.      +0.j  1.      +0.j]]

Fitted 4x4 matrix:
[[ 0.82142739  0.55031312  0.14970811  0.        ]
 [-0.48151279  0.5285377   0.69913756  0.        ]
 [ 0.30561819 -0.64637711  0.69913815 -0.        ]
 [ 0.         -0.          0.          1.        ]]

Max |U_fit - U_target| = 4.364e-07
Frobenius norm error   = 7.566e-07
